In [1]:
# !pip install firebase-admin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [firebase-admin]m [google-cloud-firestore]


In [17]:
# !pip install --upgrade google-cloud-pubsub

In [1]:
# from google.cloud import pubsub_v1

# project_id = "myproducts-488109"
# subscription_id = "my-sub"

# subscriber = pubsub_v1.SubscriberClient()
# subscription_path = subscriber.subscription_path(project_id, subscription_id)

# # 最大 N 件ずつ pull
# response = subscriber.pull(
#     request={
#         "subscription": subscription_path,
#         "max_messages": 100
#     }
# )

# for received_message in response.received_messages:
#     print(f"Message: {received_message.message.data.decode('utf-8')}")
    # ACK を返す（消さないならここで処理のみ）
    # subscriber.acknowledge(
    #     request={
    #         "subscription": subscription_path,
    #         "ack_ids": [received_message.ack_id],
    #     }
    # )

In [9]:
import os
import shutil

down_fol = 'origineImages'
up_fol = 'detectedImages'

for fol_ in [down_fol, up_fol]:
    if os.path.exists(f'./{fol_}'):
        shutil.rmtree(f'./{fol_}')
    os.mkdir(f'./{fol_}')

In [6]:
import firebase_admin
from firebase_admin import credentials, storage
from glob import glob

if not firebase_admin._apps:
    # cred = credentials.Certificate(glob('./seacrets/*-firebase-adminsdk-*.json')[0])
    firebase_admin.initialize_app(#cred, 
        options={"storageBucket": "myproducts-488109.firebasestorage.app"}
    )
bucket = storage.bucket()


In [7]:
from ultralytics import YOLO
from ultralytics.utils.plotting import Colors

model = YOLO('yolov8n.pt')

In [8]:
blob = bucket.blob(f'{down_fol}/ATfCyM3WVHe8UhaFwkyN.jpg')
blob.download_to_filename(f'./{down_fol}/ATfCyM3WVHe8UhaFwkyN.jpg')

results = model(f'./{down_fol}/ATfCyM3WVHe8UhaFwkyN.jpg',save=True)
results[0].save(f'./{up_fol}/ATfCyM3WVHe8UhaFwkyN.jpg')

labels = []
labelColors = []
colors = Colors()  # 公式パレット
for box in results[0].boxes:
    class_id = int(box.cls.item())
    cla_name = model.names[class_id]
    color = colors(class_id)  # ← これが描画に使われる色
    print("class:", class_id,',', cla_name, "RGB:", color)
    r,g,b = color
    labelColor = "#{:02x}{:02x}{:02x}".format(r,g,b)
    print(labelColor)
    labels.append(cla_name)
    labelColors.append(labelColor)
print(labels)
print(labelColors)


image 1/1 /home/jupyter/mlModelCore/origineImages/ATfCyM3WVHe8UhaFwkyN.jpg: 640x640 1 person, 1 train, 1 truck, 365.9ms
Speed: 15.6ms preprocess, 365.9ms inference, 33.1ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /home/jupyter/mlModelCore/runs/detect/predict6
class: 6 , train RGB: (255, 68, 79)
#ff444f
class: 0 , person RGB: (4, 42, 255)
#042aff
class: 7 , truck RGB: (204, 237, 0)
#cced00
['train', 'person', 'truck']
['#ff444f', '#042aff', '#cced00']


In [8]:
# from datetime import datetime
# from zoneinfo import ZoneInfo

# now = datetime.now(ZoneInfo("Asia/Tokyo"))
# print(now)

2026-03-02 20:39:52.269523+09:00
